In [1]:
# extract stream from osm

import osmnx as ox
import geopandas as gpd
import leafmap

In [2]:
######### collect streams for the four focused cities separately

place_name = "Dresden, Germany"
# place_name = "Jablonec nad Nisou, Czech Republic"
# place_name = "Poznań, Poland"
# place_name = "Senica, Slovakia"

streams= ox.features_from_place(place_name,tags={"waterway":"stream"})

city_boundary = ox.geocode_to_gdf(place_name) 

In [3]:
# set the center of the map
# identify the center coordinates of the city
center = city_boundary.centroid
center_lon = center.x.values[0]
center_lat = center.y.values[0]


In [4]:

# visualise the stream
m = leafmap.Map(center=[center_lat, center_lon], zoom=12)
m.add_gdf(streams, layer_name="All Streams", style={"color": "blue", "weight": 2})
m.add_gdf(city_boundary, layer_name="City Boundary")
m 

Map(center=[np.float64(51.06632603025163), np.float64(13.783424250027194)], controls=(ZoomControl(options=['po…

In [5]:
# set crs to meters
streams = streams.to_crs(epsg=3857)


In [6]:
type(streams)

geopandas.geodataframe.GeoDataFrame

In [7]:

#### remove overlapping
import geopandas as gp
from shapely.geometry import LineString, box
from shapely.strtree import STRtree

fishnet_boxes = []

In [ ]:

# Step 1: Create boxes along streams every 50 meters
for idx, row in streams.iterrows():
    geometry = row.geometry
    
    if isinstance(geometry, LineString):
        distance = 50
        num_points = int(geometry.length // distance)
        
        for i in range(num_points + 1):
            point = geometry.interpolate(i * distance)
            fishnet = box(point.x - 25, point.y - 25, point.x + 25, point.y + 25)
            fishnet_boxes.append(fishnet)


In [8]:
# Step 2: Remove duplicates (exact same geometry)
fishnet_gseries = gp.GeoSeries(fishnet_boxes).drop_duplicates().reset_index(drop=True)

In [9]:
# Step 3: Remove overlaps using spatial indexing (efficient method)
tree = STRtree(fishnet_gseries)
non_overlapping_boxes = []

for geom in fishnet_gseries:
    # Check intersection with already selected boxes
    if not any(geom.intersects(existing_geom) for existing_geom in non_overlapping_boxes):
        non_overlapping_boxes.append(geom)

# Convert back to GeoDataFrame
fishnet_final_gdf = gp.GeoDataFrame(geometry=non_overlapping_boxes, crs=streams.crs)



In [10]:
# Visualize the final result clearly
fig, ax = plt.subplots(figsize=(12, 8))
city_boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=2)
streams.plot(ax=ax, color="dodgerblue", linewidth=1, alpha=0.7)
fishnet_final_gdf.boundary.plot(ax=ax, color="red", linestyle='--', linewidth=1)
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.5)
ax.set_title(place_name + " with Non-overlapping Fishnet Grid", fontsize=15)
ax.axis('off')
plt.show()

NameError: name 'plt' is not defined

In [78]:
#visualise the points and lines


m = leafmap.Map(center=[center_lat, center_lon], zoom=12)
m.add_gdf(stream, layer_name="Stream", style={"color": "red", "weight": 5})
m.add_gdf(points_gdf, layer_name="Points", style={"color": "blue", "radius": 5})
m.add_gdf(city_boundary, layer_name="City Boundary")
m.add_gdf(perp_lines_gdf, layer_name="Perp Lines", style={"color": "green", "weight": 2})

m

Map(center=[np.float64(51.06632603025163), np.float64(13.783424250027194)], controls=(ZoomControl(options=['po…

In [79]:
# create buffer zones around the stream with 50m radius
buffered_stream = stream.copy()
buffered_stream["geometry"] = stream.buffer(50)


In [81]:
# visualise the buffer zones
m = leafmap.Map(center=[center_lat, center_lon], zoom=12)
m.add_gdf(stream, layer_name="Stream", style={"color": "red", "weight": 5})
m.add_gdf(buffered_stream, layer_name="Buffered Stream", style={"color": "blue", "weight": 2})
m.add_gdf(city_boundary, layer_name="City Boundary")
m.add_gdf(perp_lines_gdf, layer_name="Perp Lines", style={"color": "green", "weight": 2})
m

Map(center=[np.float64(51.06632603025163), np.float64(13.783424250027194)], controls=(ZoomControl(options=['po…